## Timestamp extraction sandbox

Play with extracting image acquisition times.

1. Read EXIF DateTimeOriginal via Pillow.
2. Fall back to filesystem `stat().st_mtime` if EXIF is missing.
3. Compare results for `.bmp` sequences and convert to seconds since epoch.


In [ ]:
from pathlib import Path
from datetime import datetime
from typing import Optional

from PIL import Image, ExifTags

def get_exif_timestamp(path: Path) -> Optional[datetime]:
    try:
        img = Image.open(path)
        info = img._getexif() or {}
    except Exception as exc:
        print('EXIF read failed', exc)
        return None
    for tag, value in info.items():
        decoded = ExifTags.TAGS.get(tag, tag)
        if decoded in ('DateTimeOriginal', 'DateTime'):
            try:
                return datetime.strptime(value, '%Y:%m:%d %H:%M:%S')
            except ValueError:
                pass
    return None

def get_filesystem_timestamp_seconds(path: Path) -> float:
    return path.stat().st_mtime

example = Path('/Volumes/Extreme SSD/deep-sea-particles-gm/pyrite_2024-11-14 20-00-15.825503_good/0/0_4021.bmp')
print('EXIF ts', get_exif_timestamp(example))
print('stat ts (datetime)', datetime.fromtimestamp(get_filesystem_timestamp_seconds(example)))
print('stat ts (seconds)', get_filesystem_timestamp_seconds(example))


In [ ]:
example = Path('/Volumes/Extreme SSD/deep-sea-particles-gm/pyrite_2024-11-14 20-00-15.825503_good/0/0_4021.bmp')
print('EXIF ts', get_exif_timestamp(example))
print('stat ts (seconds)', get_filesystem_timestamp_seconds(example))

example = Path('/Volumes/Extreme SSD/deep-sea-particles-gm/pyrite_2024-11-14 20-00-15.825503_good/0/5_4026.bmp')
print('EXIF ts', get_exif_timestamp(example))
print('stat ts (seconds)', get_filesystem_timestamp_seconds(example))


In [ ]:
# iterate over sample directory once you set `image_dir`
image_dir = Path('path/to/your/sequence')
records = []
for frame in sorted(image_dir.glob('*.bmp')):
    records.append((frame.name, get_exif_timestamp(frame), get_filesystem_timestamp_seconds(frame)))

for name, exif_ts, fs_ts in records:
    print(name, 'EXIF', exif_ts, 'fs_seconds', fs_ts)
